# 第6章 分类模型 · 参考答案



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter6/Chapter6.md]]
- [[课程/深度学习/演示代码/Chapter6.py]]
- [[课程/深度学习/课堂笔记/2026-10-12-第11节-深度学习编程与分类模型宣讲.md]]
- [[课程/深度学习/课堂笔记/2026-10-14-第12节-神经网络分类模型代码编写.md]]
- [[课程/深度学习/课堂笔记/2026-10-19-第13节-神经网络与卷积网络课程宣讲.md]]（仅分类部分，不含卷积/池化）
- [[课程/深度学习/作业/Homework03/Homework03.md]]
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]


## 模拟要求

计时 **30 分钟，100 分**；一道完整编程题、多个小问，**从导入库开始独立编写，不做代码补全**。先作答，再运行自检。题面中的接口名称仅用于统一自检，不提供实现框架。

本地 PNG 写入当前目录 `output/`（请用 pathlib 创建目录）；正式考试若题目指定 `/output/Qxxx.png`，须严格使用正式路径和文件名。练习中的输出目录是本地适配，不是正式考试环境的改动。


## Q6A 30分钟综合题：平面四区识别

**新编模拟/演示改编，依据** [[课程/深度学习/PPT/Chapter6/Chapter6.md]]“演示：四象限分类”小节、[[课程/深度学习/演示代码/Chapter6.py]]。

**共100分，30分钟，从空白独立编程。**

1. **数据与标签（20分）**：导入库、CPU单线程、torch.manual_seed(61)。标准正态顺序采样Xa6_train=(512,2)、Xa6_test=(192,2)。实现`quadrant_a6(x)`→1—4 long，按(+,+)/(-,+)/(-,-)/(+,-)编号，零属于非负侧；ya6_train/ya6_test保留1—4。
2. **网络与编码（20分）**：net_a6为2→16→4 ReLU隐藏、raw logits输出；构造训练onehot_a6=(512,4)并核对每行和1；CrossEntropyLoss目标为ya6_train−1的long。
3. **训练（30分）**：Adam(.02)，100步全批CE，记录history_a6；每步清零、新前向、loss、backward、step。禁止在CrossEntropyLoss前softmax。
4. **最终预测（15分）**：no_grad计算test_prob_a6逐行softmax，pred1_a6=argmax+1，acc_a6为与ya6_test比较的准确率；输出训练CE与准确率。
5. **图与说明（15分）**：保存output/Q6A.png，左图训练CE，右图测试点按**预测类别**着色；固定类别1/2/3/4颜色为red/green/blue/orange，画坐标轴，英文图名/轴名。写一句解释整数标签为何减一而预测展示为何加一。

不要求交互动画或GPU；性能阈值只作宽松冒烟检查，不以固定准确率代替功能评分。


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

torch.manual_seed(61)
def quadrant_a6(x):
    labels = torch.empty(len(x),dtype=torch.long)
    labels[(x[:,0]>=0)&(x[:,1]>=0)] = 1
    labels[(x[:,0]<0)&(x[:,1]>=0)] = 2
    labels[(x[:,0]<0)&(x[:,1]<0)] = 3
    labels[(x[:,0]>=0)&(x[:,1]<0)] = 4
    return labels
Xa6_train,Xa6_test = torch.randn(512,2),torch.randn(192,2)
ya6_train,ya6_test = quadrant_a6(Xa6_train),quadrant_a6(Xa6_test)
onehot_a6 = torch.zeros(512,4)
for j in range(1,5):
    onehot_a6[ya6_train==j,j-1] = 1
net_a6 = nn.Sequential(nn.Linear(2,16),nn.ReLU(),nn.Linear(16,4))
opt_a6 = torch.optim.Adam(net_a6.parameters(),lr=.02)
criterion_a6,history_a6 = nn.CrossEntropyLoss(),[]
for _ in range(100):
    opt_a6.zero_grad()
    loss = criterion_a6(net_a6(Xa6_train),(ya6_train-1).long())
    loss.backward()
    opt_a6.step()
    history_a6.append(loss.item())
net_a6.eval()
with torch.no_grad():
    test_logits_a6 = net_a6(Xa6_test)
    test_prob_a6 = torch.softmax(test_logits_a6,dim=1)
    pred1_a6 = test_logits_a6.argmax(1)+1
    acc_a6 = (pred1_a6==ya6_test).float().mean().item()
    final_ce_a6 = criterion_a6(net_a6(Xa6_train),ya6_train-1).item()
Path("output").mkdir(exist_ok=True)
fig,axes = plt.subplots(1,2,figsize=(9,3.5))
axes[0].plot(history_a6)
axes[0].set(xlabel="Update",ylabel="Training CE",title="Training loss")
for j,color in enumerate(["red","green","blue","orange"],1):
    mask = pred1_a6==j
    points = Xa6_test[mask].detach().cpu().numpy()
    axes[1].scatter(points[:,0],points[:,1],s=9,c=color,label=str(j))
axes[1].axhline(0,color="gray",lw=.5)
axes[1].axvline(0,color="gray",lw=.5)
axes[1].set(xlabel="x1",ylabel="x2",title="Predicted classes")
axes[1].legend()
fig.tight_layout()
fig.savefig("output/Q6A.png",dpi=120)
plt.close(fig)
print("Final training CE:",final_ce_a6,"Test accuracy:",acc_a6)
print("整数损失目标使用0—3下标故减一；展示课程1—4标签故预测加一。")


### 解析与点评

参数量48+68=116。one-hot用来核验编号关系，整数目标接口不必在loss里使用one-hot。分类图必须按预测类别画，不能偷偷用真标签让图看起来完美。原脚本手写概率损失与本卷库logits接口需区分。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
import torch
from pathlib import Path
assert quadrant_a6(torch.tensor([[0.,0.],[0.,-1.],[-1.,0.],[-1.,-1.]])).tolist()==[1,4,2,3]
assert ya6_train.dtype==torch.long and onehot_a6.shape==(512,4)
assert torch.equal(onehot_a6.argmax(1)+1,ya6_train)
assert sum(p.numel() for p in net_a6.parameters()) == 116
assert net_a6(torch.zeros(1,2)).shape==(1,4)
assert len(history_a6)==100 and np.isfinite(history_a6).all()
assert torch.allclose(test_prob_a6.sum(1),torch.ones(192),atol=1e-6)
assert torch.equal(test_prob_a6.argmax(1)+1,pred1_a6)
assert np.isclose(acc_a6,(pred1_a6==ya6_test).float().mean().item())
assert .4 <= acc_a6 <= 1  # smoke check only
assert Path("output/Q6A.png").stat().st_size>1000

print("自检通过")
